In [1]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install pandas
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = "data/allbirds.db"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


def main():
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("""
            SELECT product_id, variant_id, scraped_at,
                   price, available
            FROM observations
        """, conn)

    if df.empty:
        raise SystemExit("No data to analyse.")

    df["scraped_at"] = pd.to_datetime(
        df["scraped_at"], errors="coerce", utc=True
    )
    df = df.dropna(subset=["scraped_at"])
    df["date"] = df["scraped_at"].dt.date

    # Summarise daily availability
    daily = df.groupby("date").agg(
        variant_observations=("variant_id", "size"),
        available_observations=("available", "sum"),
        distinct_products=("product_id", "nunique")
    )

    daily["availability_percent"] = (
        df.groupby("date")["available"].mean() * 100
    )

    print("Daily summary:")
    print(daily)

    daily.to_csv(OUT_DIR / "daily_summary.csv")

    plt.figure()
    daily["availability_percent"].plot(marker="o")
    plt.title("Percentage of Variant Observations Available by Day")
    plt.xlabel("Collection date")
    plt.ylabel("Available observations (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / "availability_percentage_by_day.png",
        dpi=160
    )
    plt.close()

    # Compare the first and last observed status per variant.
    # Variants absent from one day's feed cannot automatically be
    # classified as out of stock.
    df = df.sort_values("scraped_at")
    df = df.dropna(subset=["variant_id", "available"])

    if not df.empty:
        df["variant_key"] = (
            df["product_id"].astype(str) + "_" +
            df["variant_id"].astype(str)
        )

        first = df.groupby("variant_key").first()["available"]
        last = df.groupby("variant_key").last()["available"]

        comparison = pd.DataFrame({
            "first_status": first,
            "last_status": last
        })

        comparison["status_changed"] = (
            comparison["first_status"] != comparison["last_status"]
        )

        print("\nFirst-to-last variant availability comparison:")
        print(comparison["status_changed"].value_counts())
        print(
            "Variants whose first and last recorded statuses differ:",
            int(comparison["status_changed"].sum())
        )


if __name__ == "__main__":
    main()


Daily summary:
            variant_observations  available_observations  distinct_products  \
date                                                                          
2026-10-15                  7454                    2346                692   
2026-10-16                 14908                    4732                692   
2026-10-17                 14908                    4682                692   
2026-10-18                  7454                    2328                692   
2026-10-19                  7454                    2318                692   

            availability_percent  
date                              
2026-10-15             31.473035  
2026-10-16             31.741347  
2026-10-17             31.405957  
2026-10-18             31.231554  
2026-10-19             31.097397  

First-to-last variant availability comparison:
status_changed
False    7384
True       70
Name: count, dtype: int64
Variants whose first and last recorded statuses differ: 70
